# 🕵️‍♂️ Gemini Deep Research Agent Lab
**Version:** 1.0 (Phase 0: Calibration)

This notebook is the dedicated laboratory for the **Agentic Research** phase of the O-ISAC Survey.
It allows running the Gemini Agent in "Systematic Reviewer" mode to analyze papers, propose search strategies, and triage results.

---

## 1. Setup & Environment
Initialize the environment, install dependencies, and load API keys.

In [15]:
# @title Install Dependencies
!pip install -q -U google-generativeai

In [16]:
# @title Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Define Root Path
BASE_DIR = "/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST"
os.chdir(BASE_DIR)
print(f"📂 Working Directory set to: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Working Directory set to: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [ ]:
# @title Load API Key
import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("🔑 Google API Key loaded successfully.")
except Exception as e:
    print(f"❌ Error loading API Key: {e}\nPlease ensure 'GOOGLE_API_KEY' is set in Colab Secrets.")

🔑 Google API Key loaded successfully.


## 2. Phase 0: Calibration (Golden Set Test)
Run the agent on `O_ISAC_029` (our Golden Sample) to generate an **Evidence Package**.
We will compare this output with our manual/pipeline results to measure Recall/Precision.

In [ ]:
# @title Run Calibration Agent
import sys

# Add analysis directory to path to allow importing modules if needed
sys.path.append(os.path.join(BASE_DIR, "analysis"))

# Run the script directly
!python "analysis/deep_res/run_calibration.py"

🕵️‍♂️ Deep Research Agent - Calibration Mode: O_ISAC_029
✅ Found paper text in: data/proc_markdowns
✅ Loaded Paper Text (79038 chars)
✅ Loaded System Role
⏳ Agent Thinking...
✅ Analysis Complete.
💾 Evidence Package saved to: analysis/deep_res/output/O_ISAC_029_DeepResearch_Evidence.md


### 📊 View Results
Once the agent finishes, run the cell below to view the generated Evidence Package.

In [ ]:
# @title Display Evidence Package
output_file = "analysis/deep_res/output/O_ISAC_029_DeepResearch_Evidence.md"

if os.path.exists(output_file):
    with open(output_file, 'r', encoding='utf-8') as f:
        content = f.read()
    print(f"\n📄 Report Content ({len(content)} chars):\n" + "="*40 + "\n")
    print(content)
else:
    print("❌ Report file not found. Did the agen run successfully?")


📄 Report Content (6879 chars):

## Evidence Package

## 1. System Classification
-   **Domain**: Hybrid (Fiber-Wireless)
-   **Coupling Mode**: Time Division Multiplexing (TDM)
-   **Waveform Relationship**: Separate (QAM for communication, LFM for sensing)

## 2. Hard Evidence (Verifiable Numbers)
-   **Comm Metrics**:
    *   **Data Rate**: 116 Gbit/s (peak), 108.4 Gbit/s (peak net rate with 7% FEC overhead for 29 GBaud 16-QAM)
    *   **BER**: < 1E-2 (for 29 GBaud 16-QAM, pre-FEC)
    *   **Distance**:
        *   **Fiber**: 20 km (SMF transmission)
        *   **Wireless**: 1 m (wireless link transmission, THz)
-   **Sensing Metrics**:
    *   **Resolution (Range)**: 6 mm (at 134 GHz, with 30 GHz LFM bandwidth)
    *   **Accuracy (Ranging Error)**: < 3 mm (calibrated mean error)
    *   **Center Frequency (THz)**: 134 GHz (optimal range 129–134 GHz)
-   **Hardware**:
    *   **Laser Type**:
        *   ECL-1: External Cavity Laser, 193.4022 THz, 100 kHz linewidth, 14.5 dBm optical

## 3. Phase 1: AI Bulk Screening (Scopus)
This section automates the screening of candidate papers using Gemini.
We load the `scopus_candidates.csv`, ask Gemini to evaluate relevance based on the Title (and Abstract if available), and output a decision.


In [17]:
# @title Load Scopus Candidates
import pandas as pd
import os

CANDIDATES_CSV = os.path.join(BASE_DIR, 'scopus_candidates.csv')
if os.path.exists(CANDIDATES_CSV):
    df_candidates = pd.read_csv(CANDIDATES_CSV)
    print(f'✅ Loaded {len(df_candidates)} candidates from {CANDIDATES_CSV}')
    display(df_candidates.head())
else:
    print(f'❌ File not found: {CANDIDATES_CSV}')


✅ Loaded 226 candidates from /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST/scopus_candidates.csv


,Track_ID,Document Title,Authors,Publication Title,Publication Year,DOI,CATEGORY
0,O_ISAC_165,Design and modelling of InAs detection system ...,H. Luo; N. Xu; S. Wang; H. Kuang; H. Ge; X. Li...,Optics Communications,2026,10.1016/j.optcom.2025.132782,WIRELESS
1,O_ISAC_166,An integrated sensing and communication solar ...,J. Li; J. Tao; C. Ge; C. Xu; J. Wang; Z. Xu; H...,Nano Energy,2026,10.1016/j.nanoen.2025.111639,WIRELESS
2,O_ISAC_167,Multi-service integrated sensing and communica...,J. Shen; J. Fan; M. Li; Y. Fei; T. Fu; M. Gao,Optics and Lasers in Engineering,2026,10.1016/j.optlaseng.2025.109470,FIBER
3,O_ISAC_168,Space-Frequency Switching MIMO-OFDM ISAC Syste...,K.I. Lee; J. Myung Shin; S. Young Park; K.W. Choi,IEEE Internet of Things Journal,2025,10.1109/JIOT.2025.3617471,WIRELESS
4,O_ISAC_169,Terahertz asynchronous twin-comb for prefiguri...,L. Ma; F. Fan; J. Feng; P. Shen; C. Song; Y. J...,Nature Communications,2025,10.1038/s41467-025-63513-z,WIRELESS


In [18]:
# @title Define Screening Agent
import json
import time
from google.generativeai.types import HarmCategory, HarmBlockThreshold
import google.generativeai as genai

def screen_paper(title, authors, year, category):
    # 1. Construct Prompt
    prompt = f"""
    Act as a Senior Systematic Reviewer for a survey on 'Optical Integrated Sensing and Communication (O-ISAC)'.

    INCLUSION CRITERIA (Must meet ALL):
    1. MUST involve BOTH 'Sensing' (Radar, Lidar, Positioning, Detection) AND 'Communication' (Data transmission).
    2. MUST involve 'Optical' technologies (Visible Light, FSO, Fiber, Photonic Integrated Circuits, etc.).
    3. Dual-functionality must be INTEGRATED (shared hardware, spectrum, or waveform).

    EXCLUSION CRITERIA (Reject if ANY):
    1. Purely RF/Millimeter-wave/THz studies without any optical/photonic component.
    2. Purely sensing (e.g., just Lidar) or purely communication (e.g., just FSO) without the other function.
    3. 'Optical Sensing' for non-comm purposes (e.g., simple temp sensor) unless integrated with comms.

    Analyze this paper:
    - Title: {title}
    - Authors: {authors}
    - Year: {year}
    - Auto-Category: {category}

    Provide a JSON response with keys:
    - 'decision': 'Included' or 'Excluded'
    - 'reason': Concise explanation (max 1 sentence) referencing the criteria.
    - 'confidence': 0.0 to 1.0
    """

    try:
        model = genai.GenerativeModel('gemini-2.5-flash')
        response = model.generate_content(
            prompt,
            generation_config={'response_mime_type': 'application/json'}
        )
        return json.loads(response.text)
    except Exception as e:
        print(f'Error: {e}')
        return {'decision': 'Error', 'reason': str(e), 'confidence': 0.0}



In [ ]:
# @title Test Screening (First 5 Rows)
results = []
print('🧪 Testing Agent on first 5 rows...')

if 'df_candidates' in locals():
    for idx, row in df_candidates.head(5).iterrows():
        print(f'Processing {idx+1}: {row["Document Title"][:50]}...')
        decision = screen_paper(
            row['Document Title'],
            row['Authors'],
            row['Publication Year'],
            row.get('CATEGORY', 'Unknown')
        )
        print(f'   👉 {decision["decision"]}: {decision["reason"]}')
        results.append(decision)
        time.sleep(1) # Rate limit politeness
else:
    print('⚠️ df_candidates not loaded. Run the generic load cell first.')


In [19]:
# @title 🛠️ Utility: List Available Groq Models
# Run this to see which models you have access to (e.g. llama3-70b-8192)
!pip install -q groq
import os
from google.colab import userdata
from groq import Groq

try:
    # Try fetching key
    api_key = None
    try: api_key = userdata.get('GROQ_API_KEY')
    except: pass
    if not api_key: api_key = os.environ.get('GROQ_API_KEY')

    if not api_key:
        print('❌ Error: GROQ_API_KEY not found in Secrets or Env')
    else:
        client = Groq(api_key=api_key)
        models = client.models.list()
        print(f'✅ Connected! Found {len(models.data)} models:')
        for m in sorted(models.data, key=lambda x: x.id):
            print(f'   - {m.id}')
except Exception as e:
    print(f'❌ Error: {e}')


✅ Connected! Found 22 models:
   - allam-2-7b
   - canopylabs/orpheus-arabic-saudi
   - canopylabs/orpheus-v1-english
   - groq/compound
   - groq/compound-mini
   - llama-3.1-8b-instant
   - llama-3.3-70b-versatile
   - meta-llama/llama-4-maverick-17b-128e-instruct
   - meta-llama/llama-4-scout-17b-16e-instruct
   - meta-llama/llama-guard-4-12b
   - meta-llama/llama-prompt-guard-2-22m
   - meta-llama/llama-prompt-guard-2-86m
   - moonshotai/kimi-k2-instruct
   - moonshotai/kimi-k2-instruct-0905
   - openai/gpt-oss-120b
   - openai/gpt-oss-20b
   - openai/gpt-oss-safeguard-20b
   - playai-tts
   - playai-tts-arabic
   - qwen/qwen3-32b
   - whisper-large-v3
   - whisper-large-v3-turbo


## 3. Phase 1: AI Bulk Screening (Scopus)
This section automates the screening of candidate papers using **Llama 3.3 70B** via Groq.
We load the `scopus_candidates.csv`, ask the model to evaluate relevance based on the Title (and Abstract if available), and output a decision.


In [20]:
# @title Load Scopus Candidates
import pandas as pd
import os

CANDIDATES_CSV = os.path.join(BASE_DIR, 'scopus_candidates.csv')
if os.path.exists(CANDIDATES_CSV):
    df_candidates = pd.read_csv(CANDIDATES_CSV)
    print(f'✅ Loaded {len(df_candidates)} candidates from {CANDIDATES_CSV}')
    display(df_candidates.head())
else:
    print(f'❌ File not found: {CANDIDATES_CSV}')


✅ Loaded 226 candidates from /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST/scopus_candidates.csv


,Track_ID,Document Title,Authors,Publication Title,Publication Year,DOI,CATEGORY
0,O_ISAC_165,Design and modelling of InAs detection system ...,H. Luo; N. Xu; S. Wang; H. Kuang; H. Ge; X. Li...,Optics Communications,2026,10.1016/j.optcom.2025.132782,WIRELESS
1,O_ISAC_166,An integrated sensing and communication solar ...,J. Li; J. Tao; C. Ge; C. Xu; J. Wang; Z. Xu; H...,Nano Energy,2026,10.1016/j.nanoen.2025.111639,WIRELESS
2,O_ISAC_167,Multi-service integrated sensing and communica...,J. Shen; J. Fan; M. Li; Y. Fei; T. Fu; M. Gao,Optics and Lasers in Engineering,2026,10.1016/j.optlaseng.2025.109470,FIBER
3,O_ISAC_168,Space-Frequency Switching MIMO-OFDM ISAC Syste...,K.I. Lee; J. Myung Shin; S. Young Park; K.W. Choi,IEEE Internet of Things Journal,2025,10.1109/JIOT.2025.3617471,WIRELESS
4,O_ISAC_169,Terahertz asynchronous twin-comb for prefiguri...,L. Ma; F. Fan; J. Feng; P. Shen; C. Song; Y. J...,Nature Communications,2025,10.1038/s41467-025-63513-z,WIRELESS


In [21]:
# @title Define Screening Agent (Llama-3.3-70b)
!pip install -q groq
import json
import time
import os
from google.colab import userdata
from groq import Groq

# Setup Groq Client
try:
    api_key = None
    try: api_key = userdata.get('GROQ_API_KEY')
    except: pass
    if not api_key: api_key = os.environ.get('GROQ_API_KEY')

    if not api_key:
        raise ValueError('GROQ_API_KEY not found in Secrets or Env')

    client = Groq(api_key=api_key)
    print('✅ Groq Client Initialized')
except Exception as e:
    print(f'❌ Setup Error: {e}')

def screen_paper(title, authors, year, category):
    # 1. Construct Prompt
    prompt = f"""
    Act as a Senior Systematic Reviewer for a survey on 'Optical Integrated Sensing and Communication (O-ISAC)'.

    INCLUSION CRITERIA (Must meet ALL):
    1. MUST involve BOTH 'Sensing' (Radar, Lidar, Positioning, Detection) AND 'Communication' (Data transmission).
    2. MUST involve 'Optical' technologies (Visible Light, FSO, Fiber, Photonic Integrated Circuits, etc.).
    3. Dual-functionality must be INTEGRATED (shared hardware, spectrum, or waveform).

    EXCLUSION CRITERIA (Reject if ANY):
    1. Purely RF/Millimeter-wave/THz studies without any optical/photonic component.
    2. Purely sensing (e.g., just Lidar) or purely communication (e.g., just FSO) without the other function.
    3. 'Optical Sensing' for non-comm purposes (e.g., simple temp sensor) unless integrated with comms.

    Analyze this paper:
    - Title: {title}
    - Authors: {authors}
    - Year: {year}
    - Auto-Category: {category}

    Provide a JSON response with keys:
    - 'decision': 'Included' or 'Excluded'
    - 'reason': Concise explanation (max 1 sentence) referencing the criteria.
    - 'confidence': 0.0 to 1.0
    """

    try:
        completion = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant that outputs strictly JSON.'},
                {'role': 'user', 'content': prompt}
            ],
            temperature=0,
            response_format={'type': 'json_object'}
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        print(f'Error: {e}')
        return {'decision': 'Error', 'reason': str(e), 'confidence': 0.0}



✅ Groq Client Initialized


In [22]:
# @title Test Screening (First 5 Rows)
results = []
print('🧪 Testing Llama-3.3 Agent on first 5 rows...')

if 'df_candidates' in locals() and 'client' in locals():
    for idx, row in df_candidates.head(5).iterrows():
        print(f'Processing {idx+1}: {row["Document Title"][:50]}...')
        decision = screen_paper(
            row['Document Title'],
            row['Authors'],
            row['Publication Year'],
            row.get('CATEGORY', 'Unknown')
        )
        print(f'   👉 {decision["decision"]}: {decision["reason"]}')
        results.append(decision)
        time.sleep(1) # Rate limit politeness
else:
    print('⚠️ df_candidates or client not loaded. Run the previous cells first.')


🧪 Testing Llama-3.3 Agent on first 5 rows...
Processing 1: Design and modelling of InAs detection system on S...
   👉 Included: The paper involves both 'sensing' and 'communication' functions integrated with optical technologies, meeting the inclusion criteria.
Processing 2: An integrated sensing and communication solar skin...
   👉 Included: The paper involves integrated sensing and communication, utilizing optical technologies for health monitoring and human-machine interaction, meeting the inclusion criteria.
Processing 3: Multi-service integrated sensing and communication...
   👉 Included: The paper involves both sensing and communication and is categorized under fiber, indicating the use of optical technologies and potential integration of dual-functionality.
Processing 4: Space-Frequency Switching MIMO-OFDM ISAC Systems: ...
   👉 Excluded: The paper focuses on RF/Millimeter-wave technologies (MIMO-OFDM) without any optical/photonic component, violating exclusion criterion 1.
Proc

In [23]:
# @title 🚀 Phase 2: RUN FULL BATCH (226 Items)
import csv
from tqdm.notebook import tqdm

OUTPUT_FILE = os.path.join(BASE_DIR, 'analysis/ph1_scr/ai_scr_dec_scopus.csv')

print(f'🚀 Starting Bulk Screening for {len(df_candidates)} papers using Llama 3.3...')

# Initialize Output File (Write Header)
with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Track_ID', 'Title', 'Decision', 'Reason', 'Confidence'])

included_count = 0

for idx, row in tqdm(df_candidates.iterrows(), total=len(df_candidates)):
    track_id = row.get('Track_ID', f'TEMP_{idx}')
    title = row['Document Title']

    # Run Screening Agent
    try:
        decision_data = screen_paper(
            title,
            row['Authors'],
            row['Publication Year'],
            row.get('CATEGORY', 'Unknown')
        )

        # Visual Feedback for Included items
        if decision_data['decision'] == 'Included':
            print(f'✅ [INCLUDED] {title[:60]}...')
            included_count += 1
        else:
            # Optional: Print excluded too if verbose, but keeping it clean
            pass

        # Save Result Immediately
        with open(OUTPUT_FILE, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                track_id,
                title,
                decision_data['decision'],
                decision_data['reason'],
                decision_data.get('confidence', 0.0)
            ])

    except Exception as e:
        print(f'❌ Error on {track_id}: {e}')

    # Rate limit politeness (Groq is fast)
    time.sleep(0.2)

print('='*40)
print(f'🏁 Batch Complete! Results saved to: {OUTPUT_FILE}')
print(f'📊 Total Included Candidates: {included_count} / {len(df_candidates)}')


🚀 Starting Bulk Screening for 226 papers using Llama 3.3...


  0%|          | 0/226 [00:00<?, ?it/s]

✅ [INCLUDED] Design and modelling of InAs detection system on SOI via mon...
✅ [INCLUDED] An integrated sensing and communication solar skin for healt...
✅ [INCLUDED] Multi-service integrated sensing and communication system fo...
✅ [INCLUDED] Frequency-comb-steered ultrawideband quasi-true-time-delay b...
✅ [INCLUDED] Joint break-point localization sensing based on OFDR and non...
✅ [INCLUDED] Poster: High-Performance Optical Camera Communications for I...
✅ [INCLUDED] Research on period-one photonic microwave scheme for integra...
✅ [INCLUDED] Scalable AI-assisted optical radio environment map estimatio...
✅ [INCLUDED] Photonics-aided integrated sensing and communication system ...
✅ [INCLUDED] Adaptive visible light integrated sensing and communication ...
✅ [INCLUDED] Chaotic optoelectronic oscillators-based dual-function radar...
✅ [INCLUDED] Photonic-aided Doppler-robust ISAC system based on index mod...
✅ [INCLUDED] Endogenous integration of communication and interference fad...

## 4. Phase 3: Extraction (O-ISAC Pipeline)
This section processes the `extraction_queue.csv` (newly included studies) to generate the final dataset.
Steps:
1. Install Dependencies (Marker for PDF->MD).
2. Convert PDFs to Markdown.
3. Run Deep Extraction with Schema v2.1.


In [ ]:
# @title 1. Install Extraction Dependencies
!pip install -q marker-pdf
!pip install -q groq
print('✅ Dependencies installed.')


In [ ]:
# @title 2. PDF to Markdown Conversion (Marker)
import os
import glob
import subprocess
from tqdm.notebook import tqdm

PDF_DIR = os.path.join(BASE_DIR, 'data/ret_docs')
MARKDOWN_DIR = os.path.join(BASE_DIR, 'data/proc_markdowns')
os.makedirs(MARKDOWN_DIR, exist_ok=True)

# Filter for papers in the queue
import pandas as pd
QUEUE_FILE = os.path.join(BASE_DIR, 'analysis/ph2_ext/extraction_queue.csv')
if os.path.exists(QUEUE_FILE):
    queue_df = pd.read_csv(QUEUE_FILE)
    target_mode = True
    target_ids = set(queue_df['Track_ID'].astype(str).str.strip())
    print(f'🎯 Targeting {len(target_ids)} papers from Queue.')
else:
    target_mode = False
    print('⚠️ Queue file not found. Processing ALL PDFs in retrieved_docs.')

pdf_files = glob.glob(os.path.join(PDF_DIR, '*.pdf'))
to_process = []

for pdf_path in pdf_files:
    pid = os.path.splitext(os.path.basename(pdf_path))[0]
    # Check if needs processing
    # 1. Is it in target list? (if mode is on)
    if target_mode and pid not in target_ids: continue
    
    # 2. output exists?
    out_dir = os.path.join(MARKDOWN_DIR, pid)
    if not os.path.exists(out_dir):
         to_process.append(pdf_path)

print(f'📋 Found {len(to_process)} PDFs to convert.')

for pdf_path in tqdm(to_process):
    pid = os.path.splitext(os.path.basename(pdf_path))[0]
    out_dir = os.path.join(MARKDOWN_DIR, pid)
    
    cmd = [
        'marker_single', pdf_path,
        '--output_dir', out_dir,
        '--paginate_output'
    ]
    # Run silently
    try:
       subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
       print(f'❌ Error converting {pid}: {e}')


### 3. Run LLM Extraction (Deep Extraction)
Loads the Markdown, feeds it to Llama 3.3 / Gemini, and fills the Schema.


In [ ]:
# @title Load Schema & Prompt
SCHEMA_FILE = os.path.join(BASE_DIR, 'analysis/oisac_ext_sch_v2.yaml')
if os.path.exists(SCHEMA_FILE):
    with open(SCHEMA_FILE, 'r') as f:
        offset = f.read()
        print('✅ Schema Loaded')
else:
    print('❌ Schema file not found!')
SYSTEM_PROMPT = '''You are a Senior Technical Editor.''' # (Simplified for brevity in injection, relying on definition in cell above or reloading from file)
# Ideally we load the extraction prompt from file too or define it here.
# Let's define the Schema injection logic here for simplicity.
pass


In [ ]:
# @title 🚀 Execute Batch Extraction Agent
import json
import time
from groq import Groq
from google.colab import userdata

# Setup Groq
try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except:
    client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

RESULTS_FILE = os.path.join(BASE_DIR, 'analysis/ph2_ext/extraction_dataset.csv')
# Logic to read markdown, prompt LLM, save row to CSV
# ... (Simplified placeholder for the full logic found in pipeline_v3)
print('⚠️  NOTE: Ensure you have copied the full extraction logic from extraction_pipeline_v3.py if this cell is not enough.')
